1. 此代码仅用于学习与参考，禁止二改二传禁止商用。
2. 更具体的数据分析与业务分析参考文档见github账号：wangziyong0315
3. 后续待补充。

In [7]:
# 二手房数据预处理部分
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/dataml999/second-hand-housing-raw/gy_data.csv')
# print(df.dtypes)
# df.columns

#  Index(['房源ID', '地址', '价格(万)', '单价', '客厅', '卧室A', '卧室B', '厨房', '卫生间', '阳台',
#        '房屋户型', '所在楼层', '建筑面积', '户型结构', '套内面积', '建筑类型', '房屋朝向', '建筑结构', '装修情况',
#        '梯户比例', '配备电梯', '产权年限', '挂牌时间', '交易权属', '上次交易', '房屋用途', '房屋年限', '产权所属',
#        '抵押信息', '房本备件', '房源描述'],
#       dtype='object')

## 删掉不要的变量（列）
df.drop(['客厅', '卧室A', '卧室B', '厨房', '卫生间','阳台'], axis=1, inplace=True)
df.drop(['房源描述', '房本备件', '房屋年限', '上次交易', '房源ID'], axis=1, inplace=True)   #  axis的1表示列2表示行

## 判断价格是否有缺失，查看缺失的数量
sum(df['价格(万)'].isnull())

## 用均值填充价格中缺失的部分
m = round(df['价格(万)'].mean(), 2)  # round保留小数点，2表示保留小数点后两位
df['价格(万)'].fillna(m)   # fillna表示填充缺失值

df['单价'] = df['单价'].str.replace('元/平米','').astype('float')
df['单价'].fillna(round(df['单价'].mean(), 2))  # 均值填充缺失值

## 计算房屋户型评率
df['房屋户型'].value_counts()

## 用高频率出现的数据填充进null
df['房屋户型'] = df['房屋户型'].fillna('3室2厅1厨2卫')

## 把房屋户型的 室厅厨卫 提取出来并创建新的列：房间数，客厅数，厨房数，卫生间数
hx = df['房屋户型'].str.replace('室|厅|厨|卫', '-', regex=True).str.split('-', expand=True) # regex是正则表达式(书上34页) ，|是或的意思
df['房间数'] = hx[0].astype('int')
df['客厅数'] = hx[1].astype('int')
df['厨房数'] = hx[2].astype('int')
df['卫生间数'] = hx[3].astype('int')

## 删除房屋户型
df.drop('房屋户型', axis=1, inplace=True)

## 同样的方法处理所在楼层和建筑面积，要求查看所在楼层有多少缺失值，新增两列：所处楼层和楼层数
# sum(df['所在楼层'].isnull())
# df['所在楼层'].value_counts()
# df['所在楼层'].fillna('中楼层 (共8层) ')
lc = df['所在楼层'].str.replace('未知(共0层)', '中楼层(共8层)').str.replace('楼层|(共|层)|下室', '-', 
                                                               regex=True).str.split('-', expand=True)

# df['所在楼层'].value_counts()
# lc[0] = lc[0].fillna(lc[0].value_counts()[0])
# lc[2] = lc[2].fillna(lc[2].value_counts()[0])
# 新版本代码如下
lc[0] = lc[0].fillna(lc[0].value_counts().iloc[0])
lc[2] = lc[2].fillna(lc[2].value_counts().iloc[0])

df['所处楼层'] = lc[0]
df['楼层数'] = lc[2]

sum(df['建筑面积'].isnull())
df['建筑面积'] = df['建筑面积'].str.replace('㎡','').astype('float')
df['建筑面积'].fillna(round(df['建筑面积'].mean(), 2))

## 查看 数据量 和 数据类型
# df.info()

## 批量填充
# for col in range(4,16+1):
#     df[df.columns[col]] = df[df.columns[col]].fillna(df[df.columns[col]].value_counts()[0])
# df['地址'] = df['地址'].fillna(df['地址'].value_counts()[0])

## 批量填充用前面的值填充后面空缺的值
# df = df.fillna(method='ffill')
df = df.ffill()   # 等价于 fillna(method='ffill')

## 批量填充用后面的值填充前面空缺的值
# df = df.fillna(method='bfill')

# df['所处楼层'] = df['所处楼层'].map({'低':1, '中':2, '高':3})  # 用pandas的map函数实现列中的数值替换要用{}括在（）里面

## 简单的数据分析
df.describe()  # 查看数据的基本情况
df.groupby('装修情况')['价格(万)'].mean()  # 查看某一列数据对应另一列数据的平均值
df.groupby('房屋用途')['价格(万)'].mean() 
df.groupby(['房屋用途','装修情况'])['价格(万)'].mean()  # 查看某两列数据对应另一列数据的平均值

# # 提取 所处区域
df['所处区域'] = df['地址'].str.split('/', expand=True)[1].replace(' ', '其他区域')
df['小区'] = df['地址'].str.split('/', expand=True)[2]
df.groupby(['所处区域','小区'])['价格(万)'].mean()

## 用表格的方式提取两列数据对应的第三数据的平均值
pt = df.pivot_table(values='价格(万)', index=['所处区域','小区'])
pt.to_csv('AAA.csv',encoding='utf_8_sig')
df.to_csv('AAAA.csv',encoding='utf_8_sig')

In [14]:
# 线性回归建模部分
# 在进行建模之前要先把上面处理好的数据集AAAA.csv保存下来, 再上传到右边的upload以便引用数据集，否则会报错
# 导入需要用到的包
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 固定随机种子，方便复现
RANDOM_STATE = 42

## 读取数据
df = pd.read_csv('/kaggle/input/datasets/yong0316/clean-data/AAAA.csv')
print(f"数据大小: {df.shape}")
print("--------------------分割线------------------------")

## 确定特征列和目标列
target = '价格(万)'
num_cols = ['建筑面积', '房间数', '客厅数', '厨房数', '卫生间数', '楼层数']
cat_cols = ['所处区域', '装修情况', '所处楼层']
# 检查列是否存在
for col in num_cols + cat_cols + [target]:
    if col not in df.columns:
        raise ValueError(f"缺少列: {col}")
        
print("--------------------分割线------------------------")

## 清洗数值列：强制转为数值，无法转换的用中位数填充
print("清洗数值列...")
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    missing = df[col].isnull().sum()
    if missing > 0:
        df[col] = df[col].fillna(df[col].median())   
        print(f"  {col} 填充了 {missing} 个缺失值")

# 目标列也做同样处理
df[target] = pd.to_numeric(df[target], errors='coerce')
if df[target].isnull().any():
    df[target] = df[target].fillna(df[target].median())   

## 剔除极端价格（可选）
Q1 = df[target].quantile(0.25)
Q3 = df[target].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 3 * IQR
upper = Q3 + 3 * IQR
df = df[(df[target] >= lower) & (df[target] <= upper)]
print(f"剔除极端价格后数据量: {len(df)}")
print("--------------------分割线------------------------")

## 划分特征和标签
X = df[num_cols + cat_cols]
y = df[target]

## 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"训练集: {len(X_train)} 条, 测试集: {len(X_test)} 条")
print("--------------------分割线------------------------")

##  预处理 + 线性回归管道
# 对分类变量做独热编码，数值变量保持不变
preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

model = Pipeline([
    ('prep', preprocessor),
    ('lr', LinearRegression())
])

print("--------------------分割线------------------------")

## 训练模型
print("训练线性回归...")
model.fit(X_train, y_train)

print("--------------------分割线------------------------")

## 预测并评估
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f} 万元")
print(f"RMSE: {rmse:.2f} 万元")
print(f"R²: {r2:.4f}")

print("--------------------分割线------------------------")

## 查看系数
# 获取编码后的特征名
cat_feat_names = model.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feat_names = num_cols + list(cat_feat_names)
coefs = model.named_steps['lr'].coef_
intercept = model.named_steps['lr'].intercept_

coef_df = pd.DataFrame({'特征': all_feat_names, '系数': coefs})
coef_df = coef_df.sort_values('系数', ascending=False)
print("影响最大的前10个特征（系数最大）:")
print(coef_df.head(10))
print(f"截距: {intercept:.2f}")

print("--------------------分割线------------------------")

## 保存预测结果
result = X_test.copy()
result['真实价格'] = y_test
result['预测价格'] = y_pred
result.to_csv('线性回归算法.csv', index=False, encoding='utf_8_sig')
print("预测结果已保存到 线性回归算法.csv")

数据大小: (20059, 28)
--------------------分割线------------------------
--------------------分割线------------------------
清洗数值列...
  楼层数 填充了 1 个缺失值
剔除极端价格后数据量: 19489
--------------------分割线------------------------
训练集: 15591 条, 测试集: 3898 条
--------------------分割线------------------------
--------------------分割线------------------------
训练线性回归...
--------------------分割线------------------------
MAE: 24.64 万元
RMSE: 34.89 万元
R²: 0.5656
--------------------分割线------------------------
影响最大的前10个特征（系数最大）:
           特征          系数
19  所处楼层_7122  123.187055
13  所处区域_观山湖区   45.510048
7    所处区域_云岩区   25.993469
1         房间数   21.669892
4        卫生间数   19.819751
8    所处区域_南明区   16.920226
11   所处区域_白云区   12.848932
12   所处区域_花溪区   11.558418
6    所处区域_乌当区   11.075996
2         客厅数    4.882480
截距: 16.48
--------------------分割线------------------------
预测结果已保存到 线性回归算法.csv


In [17]:
# 随机森林建模部分

# 导入需要的包
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 固定随机种子，方便复现
RANDOM_STATE = 42

## 读数据
print("读数据...")
df = pd.read_csv('/kaggle/input/datasets/yong0316/clean-data/AAAA.csv')
print(f"数据量: {df.shape}")

print("--------------------分割线------------------------")

## 特征列
target = '价格(万)'
num_cols = ['建筑面积', '房间数', '客厅数', '厨房数', '卫生间数', '楼层数']
cat_cols = ['所处区域', '装修情况', '所处楼层']

for col in num_cols + cat_cols + [target]:
    if col not in df.columns:
        raise ValueError(f"列 {col} 不存在")

print("--------------------分割线------------------------")

## 清洗数值列
print("清洗数值列...")
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    missing = df[col].isnull().sum()
    if missing > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f"  {col} 填充 {missing} 个")

df[target] = pd.to_numeric(df[target], errors='coerce')
if df[target].isnull().any():
    df[target] = df[target].fillna(df[target].median())

print("--------------------分割线------------------------")

## 剔除太离谱的价格（可选）
Q1 = df[target].quantile(0.25)
Q3 = df[target].quantile(0.75)
IQR = Q3 - Q1
low = Q1 - 3 * IQR
high = Q3 + 3 * IQR
df = df[(df[target] >= low) & (df[target] <= high)]
print(f"剔除极端价后剩 {len(df)} 条")

print("--------------------分割线------------------------")

## 特征和标签
X = df[num_cols + cat_cols]
y = df[target]

print("--------------------分割线------------------------")

## 划分训练测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"训练: {len(X_train)}, 测试: {len(X_test)}")

print("--------------------分割线------------------------")

## 预处理 + 随机森林管道
# 分类变量做独热编码（随机森林也能用，简单易操作）
preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

model = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

print("--------------------分割线------------------------")

## 训练
print("训练随机森林...")
model.fit(X_train, y_train)

## 预测评估
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f} 万元")
print(f"RMSE: {rmse:.2f} 万元")
print(f"R²: {r2:.4f}")

print("--------------------分割线------------------------")

## 特征重要性
cat_feat_names = model.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feat_names = num_cols + list(cat_feat_names)
importances = model.named_steps['rf'].feature_importances_

imp_df = pd.DataFrame({'特征': all_feat_names, '重要性': importances})
imp_df = imp_df.sort_values('重要性', ascending=False)
print("特征重要性前10:")
print(imp_df.head(10))

print("--------------------分割线------------------------")

## 保存预测结果
result = X_test.copy()
result['真实价格'] = y_test
result['预测价格'] = y_pred
result.to_csv('随机森林模型.csv', index=False, encoding='utf_8_sig')
print("预测结果已存 随机森林模型.csv")

读数据...
数据量: (20059, 28)
--------------------分割线------------------------
--------------------分割线------------------------
清洗数值列...
  楼层数 填充 1 个
--------------------分割线------------------------
剔除极端价后剩 19489 条
--------------------分割线------------------------
--------------------分割线------------------------
训练: 15591, 测试: 3898
--------------------分割线------------------------
--------------------分割线------------------------
训练随机森林...
MAE: 18.96 万元
RMSE: 28.81 万元
R²: 0.7037
--------------------分割线------------------------
特征重要性前10:
           特征       重要性
0        建筑面积  0.776394
5         楼层数  0.093074
13  所处区域_观山湖区  0.040481
7    所处区域_云岩区  0.011423
1         房间数  0.011342
18    装修情况_精装  0.009283
4        卫生间数  0.009016
21     所处楼层_低  0.007000
16    装修情况_毛坯  0.006833
17    装修情况_简装  0.006433
--------------------分割线------------------------
预测结果已存 随机森林模型.csv
